In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, warnings, os
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize':(14,5),'font.size':12})
OUTPUT_DIR = './data'
FIGURE_DIR = './figures/nb04b'
os.makedirs(FIGURE_DIR, exist_ok=True)
panel = pd.read_parquet(os.path.join(OUTPUT_DIR,'panel_ca_daily.parquet'))
panel['date'] = pd.to_datetime(panel['date'])
print(f"Panel: {panel.shape[0]:,} x {panel.shape[1]}")

Panel: 703,087 x 44


In [2]:
MOB_CLEAN = os.path.join(OUTPUT_DIR,'chicago_mobility_daily.csv')
if os.path.exists(MOB_CLEAN):
    mobility = pd.read_csv(MOB_CLEAN, parse_dates=['date'])
    print(f"Loaded clean mobility: {len(mobility)} rows")
else:
    MOB_RAW = os.path.join(OUTPUT_DIR,'Global_Mobility_Report.csv')
    print("Filtering Cook County...")
    chunks = []
    for chunk in pd.read_csv(MOB_RAW, chunksize=500000, low_memory=False):
        m = ((chunk['country_region']=='United States') &
             (chunk['sub_region_1']=='Illinois') &
             (chunk['sub_region_2']=='Cook County'))
        if m.any():
            chunks.append(chunk[m])
    mob_raw = pd.concat(chunks, ignore_index=True)
    cmap = {
        'date':'date',
        'retail_and_recreation_percent_change_from_baseline':'mob_retail',
        'grocery_and_pharmacy_percent_change_from_baseline':'mob_grocery',
        'parks_percent_change_from_baseline':'mob_parks',
        'transit_stations_percent_change_from_baseline':'mob_transit',
        'workplaces_percent_change_from_baseline':'mob_workplace',
        'residential_percent_change_from_baseline':'mob_residential'
    }
    mobility = mob_raw[list(cmap.keys())].rename(columns=cmap)
    mobility['date'] = pd.to_datetime(mobility['date'])
    mobility = mobility.sort_values('date').reset_index(drop=True)
    mobility.to_csv(MOB_CLEAN, index=False)
    print(f"Saved: {len(mobility)} rows")

print(f"Mobility: {mobility['date'].min().date()} to {mobility['date'].max().date()}")
for c in [x for x in mobility.columns if x.startswith('mob_')]:
    print(f"  {c:<20s} nulls={mobility[c].isna().sum()}  [{mobility[c].min():+.0f}, {mobility[c].max():+.0f}]")

Filtering Cook County...
Saved: 974 rows
Mobility: 2020-02-15 to 2022-10-15
  mob_retail           nulls=0  [-81, +17]
  mob_grocery          nulls=0  [-59, +43]
  mob_parks            nulls=0  [-64, +210]
  mob_transit          nulls=0  [-74, +11]
  mob_workplace        nulls=0  [-84, +6]
  mob_residential      nulls=0  [-5, +30]


In [3]:
covid = None
for f in os.listdir(OUTPUT_DIR):
    if 'covid' in f.lower() or 'COVID' in f or 'naz8' in f:
        path = os.path.join(OUTPUT_DIR, f)
        covid_raw = pd.read_csv(path)
        print(f"Loaded: {f} ({covid_raw.shape})")
        date_col = [c for c in covid_raw.columns if 'date' in c.lower()][0]
        covid = pd.DataFrame({'date': pd.to_datetime(covid_raw[date_col])})
        for col in covid_raw.columns:
            cl = col.lower()
            if 'cases' in cl and 'total' not in cl and 'cumul' not in cl:
                covid['covid_cases'] = pd.to_numeric(covid_raw[col], errors='coerce')
            elif 'death' in cl and 'total' not in cl and 'cumul' not in cl:
                covid['covid_deaths'] = pd.to_numeric(covid_raw[col], errors='coerce')
            elif 'hosp' in cl and 'total' not in cl and 'cumul' not in cl:
                covid['covid_hosp'] = pd.to_numeric(covid_raw[col], errors='coerce')
        for c in ['covid_cases','covid_deaths','covid_hosp']:
            if c in covid.columns:
                covid[c] = covid[c].fillna(0).astype(int)
                covid[f'{c}_7d'] = covid[c].rolling(7, min_periods=1).mean()
        covid = covid.sort_values('date').reset_index(drop=True)
        print(f"Clean: {len(covid)} rows")
        break

if covid is None:
    print("COVID not found. Download from:")
    print("https://data.cityofchicago.org/api/views/naz8-j4nc/rows.csv?accessType=DOWNLOAD")

Loaded: COVID-19_Daily_Cases__Deaths__and_Hospitalizations_-_Historical.csv ((1540, 58))
Clean: 1540 rows


In [6]:
unemp = None
for f in os.listdir(OUTPUT_DIR):
    fl = f.lower()
    if 'unemp' in fl or 'seriesreport' in fl or 'bls' in fl or 'laumt' in fl:
        path = os.path.join(OUTPUT_DIR, f)
        
        # Read raw to find where actual data starts
        raw_lines = open(path, 'r').readlines()
        skip_rows = 0
        for i, line in enumerate(raw_lines):
            if 'Year' in line and 'Jan' in line:
                skip_rows = i
                break
        
        print(f"Found data header at row {skip_rows}")
        unemp_raw = pd.read_csv(path, skiprows=skip_rows)
        print(f"Shape: {unemp_raw.shape}")
        print(f"Columns: {unemp_raw.columns.tolist()}")
        print(unemp_raw.head())
        
        month_cols = ['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec']
        available_months = [m for m in month_cols if m in unemp_raw.columns]
        
        rows = []
        for _, row in unemp_raw.iterrows():
            try:
                yr = int(row['Year'])
            except:
                continue
            for m_name in available_months:
                val = pd.to_numeric(str(row[m_name]).strip(), errors='coerce')
                if pd.notna(val):
                    rows.append({'year': yr, 'month': month_cols.index(m_name)+1,
                                'unemployment_monthly': val})
        
        unemp = pd.DataFrame(rows).sort_values(['year','month']).reset_index(drop=True)
        print(f"\nClean: {len(unemp)} months")
        print(f"Range: {unemp['year'].min()}-{unemp['year'].max()}")
        print(f"Unemployment: [{unemp['unemployment_monthly'].min():.1f}%, "
              f"{unemp['unemployment_monthly'].max():.1f}%]")
        break

if unemp is None or len(unemp) == 0:
    print("Could not parse unemployment data.")

Found data header at row 10
Shape: (11, 13)
Columns: ['Year', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
   Year  Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec
0  2015  7.0  6.7  6.3  5.7  5.9  6.3  6.1  5.8  5.3  5.5  5.7  5.8
1  2016  6.6  6.6  6.4  6.0  5.5  6.2  5.8  5.8  5.7  5.6  5.1  5.1
2  2017  5.7  5.3  4.8  4.5  4.5  5.2  5.1  5.3  4.8  4.7  4.5  4.4
3  2018  5.0  4.8  4.2  3.7  3.6  4.6  4.3  4.1  3.8  3.9  3.7  4.0
4  2019  4.9  4.5  4.3  3.8  3.5  4.2  4.1  3.8  3.4  3.4  3.2  3.2

Clean: 131 months
Range: 2015-2025
Unemployment: [3.2%, 18.6%]


In [7]:
n0 = len(panel)

# Mobility
panel = panel.merge(mobility, on='date', how='left')
assert len(panel) == n0
print(f"Mobility: {panel['mob_retail'].notna().sum():,}/{n0:,}")

# COVID
if covid is not None:
    panel = panel.merge(covid, on='date', how='left')
    assert len(panel) == n0
    for c in [x for x in panel.columns if x.startswith('covid_')]:
        panel[c] = panel[c].fillna(0)
    print(f"COVID: merged")

# Unemployment
if unemp is not None:
    panel = panel.merge(unemp, on=['year','month'], how='left')
    assert len(panel) == n0
    print(f"Unemployment: {panel['unemployment_monthly'].notna().sum():,}/{n0:,}")

# Composite mobility
if 'mob_retail' in panel.columns:
    panel['mob_composite'] = panel[['mob_retail','mob_transit','mob_workplace']].mean(axis=1)

# Holiday
def is_holiday(d):
    m, day, dow = d.month, d.day, d.dayofweek
    if (m,day) in [(1,1),(7,4),(12,25),(12,31),(11,11)]: return 1
    if m==11 and dow==3 and 22<=day<=28: return 1
    if m==5 and dow==0 and day>=25: return 1
    if m==9 and dow==0 and day<=7: return 1
    if m==1 and dow==0 and 15<=day<=21: return 1
    return 0

panel['is_holiday'] = panel['date'].apply(is_holiday)
print(f"Holidays: {panel['is_holiday'].sum():,} rows")

Mobility: 74,998/703,087
COVID: merged
Unemployment: 306,999/703,087
Holidays: 17,325 rows


In [8]:
print("="*60)
print("PANEL v2 VALIDATION")
print("="*60)
print(f"Shape: {panel.shape[0]:,} x {panel.shape[1]}")

n_ca = panel['community_area'].nunique()
n_days = panel['date'].nunique()
assert panel.shape[0] == n_ca * n_days
print(f"PASS: {n_ca} CA x {n_days} days = {n_ca*n_days:,}")

print(f"\nColumns ({panel.shape[1]}):")
for i, col in enumerate(panel.columns, 1):
    nn = panel[col].isna().sum()
    s = "OK" if nn==0 else f"{nn:,} nulls ({nn/len(panel)*100:.1f}%)"
    print(f"  {i:>2d}. {col:<30s} {str(panel[col].dtype):<15s} {s}")

OUT = os.path.join(OUTPUT_DIR, 'panel_ca_daily_v2.parquet')
panel.to_parquet(OUT, index=False, engine='pyarrow')
mb = os.path.getsize(OUT)/(1024**2)
print(f"\nExported: {OUT}")
print(f"  Size: {mb:.0f} MB | Rows: {panel.shape[0]:,} | Cols: {panel.shape[1]}")

PANEL v2 VALIDATION
Shape: 703,087 x 59
PASS: 77 CA x 9131 days = 703,087

Columns (59):
   1. community_area                 int64           OK
   2. date                           datetime64[ns]  OK
   3. crime_total                    int64           OK
   4. crime_violent                  int64           OK
   5. crime_property                 int64           OK
   6. crime_theft                    int64           OK
   7. crime_battery                  int64           OK
   8. crime_homicide                 int64           OK
   9. crime_burglary                 int64           OK
  10. crime_mvt                      int64           OK
  11. crime_narcotics                int64           OK
  12. crime_robbery                  int64           OK
  13. crime_assault                  int64           OK
  14. arrest_count                   int64           OK
  15. domestic_count                 int64           OK
  16. arrest_rate                    float64         OK
  17. year     